# Session 8 — Chi-Square Test

**Goal:** test the categorical inputs — `cp`, `thal`, `slope`, `restecg`, `exang` —
where a mean is meaningless and the comparison is between whole distributions across
categories. Plus the assumption check that decides whether the test is trustworthy at
all, an effect size, and residual analysis to find *which category* drives an
association.

## What this stage does for the system

Session 5 warned that Pearson's r on `thal` (coded 3/6/7) and `cp` (coded 1-4) measures
whether an arbitrary numeric ordering happens to line up with disease — reorder the
codes and the correlation changes while the data does not. Yet those two inputs top
every ranking in the module. So they need an instrument that does not assume an
ordering, and the chi-square test of independence is it: it compares the *observed*
cross-tabulation against what would be expected if the two variables were unrelated,
and is invariant to how the categories are labelled.

The failure mode specific to this test is a quiet one. Chi-square's p-value comes from
an approximation that needs sufficiently large expected counts in every cell, and with
a rare category it silently stops being valid — no warning, just a wrong number.
`restecg` in this registry has exactly that problem, and Step 5 walks through it.

## The dataset

Every session in this module works on one registry: the UCI **Heart Disease**
dataset (Cleveland), fetched live from the UCI ML Repository with `ucimlrepo` so the
notebooks are runnable by anyone without a CSV sitting on their machine. It holds 303
patients with clinical measurements (`age`, `trestbps` resting blood pressure, `chol`
serum cholesterol, `thalach` max heart rate achieved, `oldpeak` ST depression),
categorical findings (`sex`, `cp` chest-pain type, `fbs` fasting blood sugar > 120,
`restecg`, `exang` exercise-induced angina, `slope`, `ca`, `thal`), and the outcome
`num` — angiographic disease severity 0-4, which this module binarises into
`target` (0 = no disease, 1 = disease present).

Deliberately one dataset throughout: switching datasets between topics would mean
re-learning the data every session instead of building cumulative familiarity with
one problem, the way a real analyst does.

## How to read this notebook

Every code cell is followed by a short **Observe / Infer** note: *Observe* points at
exactly what to look at in that cell's output, and *Infer* explains what conclusion to
draw from it — and what a different result would imply. Read them before running the
next cell; several of them flag things worth double-checking before you move on.

## Prerequisites

This session runs entirely locally — no account or credentials needed.

```bash
pip install ucimlrepo pandas numpy scipy scikit-learn statsmodels matplotlib seaborn
```

## Step 1 — Load the registry from the UCI repository

Fetching directly from the UCI ML Repository keeps this notebook runnable by anyone,
instead of depending on a CSV already sitting on your machine. The same nine lines
open every session in this module, so the 297 patients below are the identical 297
patients every other notebook analyses.

In [ ]:
from ucimlrepo import fetch_ucirepo
import pandas as pd

heart_disease = fetch_ucirepo(id=45)
df = pd.concat([heart_disease.data.features, heart_disease.data.targets], axis=1)

# `num` is severity 0-4; this module screens for disease presence, so binarise it.
df = df.dropna().reset_index(drop=True)
df["target"] = (df["num"] > 0).astype(int)
df = df.drop(columns="num")

print(f"{len(df)} patients, {len(df.columns)} columns")
print(f"disease prevalence: {df['target'].mean():.3f}")
df.head()

**Observe:** `297 patients, 14 columns` and `disease prevalence: 0.461`. The preview
shows `age`, `sex`, `cp`, `trestbps`, `chol`, `fbs`, `restecg`, `thalach`, `exang`,
`oldpeak`, `slope`, `ca`, `thal`, and the `target` column just derived.
**Infer:** 303 rows are fetched and 297 survive `dropna()` — six patients are missing
`ca` (number of major vessels seen on fluoroscopy) or `thal`. Dropping six rows out of
303 is defensible here and keeps every notebook in this module working on the identical
297 patients; on a larger fraction of missing values you would have to impute instead,
and *that* choice would itself need the distribution work of Session 3. If your row
count is not 297, you are on a different subset than every number quoted below.

## Step 2 — The contingency table

Chest-pain type (`cp`): 1 = typical angina, 2 = atypical angina, 3 = non-anginal pain,
4 = asymptomatic. Cross-tabulated against the outcome, this is the whole input to the
test — a chi-square sees nothing but these counts.

In [ ]:
ALPHA = 0.05

contingency = pd.crosstab(df["cp"], df["target"])
contingency.columns = ["no disease", "disease"]
contingency.index = ["1 typical angina", "2 atypical angina",
                     "3 non-anginal pain", "4 asymptomatic"]
print(contingency)

rates = contingency["disease"] / contingency.sum(axis=1)
print("\ndisease rate by chest-pain type:")
print(rates.round(3).to_string())
print(f"\nbaseline rate: {df['target'].mean():.3f}")

**Observe:** types 1-3 sit at disease rates of `0.304`, `0.184`, `0.217` — all *below*
the `0.461` baseline — while type 4 (asymptomatic) sits at `0.725` on 142 patients,
nearly half the registry.
**Infer:** the association is real but not remotely ordinal, which is exactly Session
5's warning made visible: the rate goes down from type 1 to type 2, back up at type 3,
then jumps at type 4. Any method treating `cp` as a number would try to fit a straight
line through that pattern and fail. The clinical reading is worth pausing on — *absence*
of chest pain is the strongest single indicator of disease here — which sounds backwards
until you remember these are patients already referred for angiography: something else
prompted the referral, and silent disease is the dangerous kind. It is also a reminder
that Session 1's referral-population caveat shapes what every finding in this module
means.

## Step 3 — Expected counts and the chi-square statistic

Under $H_0$ (the two variables are independent), the expected count in each cell is
`row total × column total / grand total`. The statistic sums the squared gap between
observed and expected, scaled by expected:
$\chi^2 = \sum (O - E)^2 / E$.

In [ ]:
from scipy import stats

chi2, p_value, dof, expected = stats.chi2_contingency(contingency)

print("expected counts under H0 (independence):")
print(pd.DataFrame(expected, index=contingency.index,
                   columns=contingency.columns).round(1))
print(f"\nchi-square = {chi2:.2f}")
print(f"degrees of freedom = {dof}   ((rows-1) x (cols-1))")
print(f"p = {p_value:.2e}")
print(f"\nreject H0 at alpha={ALPHA}? {p_value < ALPHA}")

**Observe:** `chi-square = 77.28` on 3 degrees of freedom, `p = 1.18e-16`, and expected
counts ranging from `10.6` up to `76.5`.
**Infer:** chest-pain type and disease are emphatically not independent. Note what the
statistic does and does not carry: `77.28` is not on any interpretable scale — it grows
with both the strength of the association *and* the sample size, which is Session 6's
significance-versus-importance problem in a new costume, and Step 6's Cramér's V is the
fix. Note also that the test is symmetric: it establishes an association between `cp`
and `target` without any notion of which is the outcome, so the causal caution from
Session 5 applies unchanged. Every expected count comfortably clears 10, which is the
condition the next steps put under stress.

## Step 4 — A 2×2 table, and the corrections that apply only there

`exang` (exercise-induced angina) is binary, so its table is 2×2 — the one shape with
special options: Yates's continuity correction (scipy's default here) and Fisher's
exact test, which computes the probability directly instead of approximating it.

In [ ]:
exang_table = pd.crosstab(df["exang"], df["target"])
exang_table.index = ["no exercise angina", "exercise angina"]
exang_table.columns = ["no disease", "disease"]
print(exang_table)
print(f"\ndisease rate: {exang_table.iloc[0, 1] / exang_table.iloc[0].sum():.3f} "
      f"vs {exang_table.iloc[1, 1] / exang_table.iloc[1].sum():.3f}")

chi2_y, p_y, _, exp_e = stats.chi2_contingency(exang_table)                    # Yates by default
chi2_n, p_n, _, _ = stats.chi2_contingency(exang_table, correction=False)
odds_ratio, p_fisher = stats.fisher_exact(exang_table.to_numpy())

print(f"\nchi-square with Yates correction: {chi2_y:.2f}, p={p_y:.2e}")
print(f"chi-square without correction:    {chi2_n:.2f}, p={p_n:.2e}")
print(f"Fisher's exact test:              p={p_fisher:.2e}, odds ratio={odds_ratio:.2f}")
print(f"smallest expected count: {exp_e.min():.1f}")

**Observe:** disease rates of `0.315` versus `0.763`, all three p-values within a
factor of three of each other around `1e-13`, and an odds ratio of `7.00`.
**Infer:** the odds ratio is the number to report — "patients with exercise-induced
angina have seven times the odds of disease" is a sentence that survives leaving the
notebook, which `chi2 = 50.94` is not. The three tests agreeing is expected here
because the smallest expected count is `44.7`, far above where the approximation
strains; Yates's correction and Fisher's exact test exist for the *opposite* case, and
their agreement now is a check rather than a finding. Note Yates is conservative (its
p-value is the largest of the three) and scipy applies it silently to every 2×2 table,
which matters if you are comparing your output against another tool's.

## Step 5 — The expected-count rule, and an input that breaks it

The chi-square p-value comes from an approximation that requires all expected counts
above 5 (a common relaxation: 80% of cells above 5, none below 1). Below that, the
p-value is simply wrong — and nothing raises an error. `restecg` is the case in this
registry.

In [ ]:
restecg_table = pd.crosstab(df["restecg"], df["target"])
restecg_table.index = ["0 normal", "1 ST-T abnormality", "2 LV hypertrophy"]
restecg_table.columns = ["no disease", "disease"]
print(restecg_table)

chi2_r, p_r, dof_r, exp_r = stats.chi2_contingency(restecg_table)
print("\nexpected counts:")
print(pd.DataFrame(exp_r, index=restecg_table.index, columns=restecg_table.columns).round(2))
print(f"\nsmallest expected count: {exp_r.min():.2f}")
print(f"cells below 5: {(exp_r < 5).sum()} of {exp_r.size}")
print(f"chi-square p = {p_r:.4f}   <- reported, but NOT trustworthy")

**Observe:** category 1 has **four patients in total**, giving expected counts of `1.85`
and `2.15` — two of six cells below 5 — yet `chi2_contingency` returns
`p = 0.0083` without complaint.
**Infer:** this is the quiet failure this session exists to prevent. A p-value under the
threshold, no warning, and a number that would sail into any results table — except the
approximation it comes from is not valid at these counts, and the true p-value could
sit on either side of 0.05. The general lesson generalises past this test: **always
inspect the expected counts, not just the p-value**, because a statistical function's
willingness to return a number is not evidence the number means anything. Three fixes
follow, in order of preference.

In [ ]:
# Fix 1: collapse rare categories into a clinically coherent group.
collapsed = pd.crosstab(df["restecg"].replace({1: 2}), df["target"])
collapsed.index = ["0 normal", "1+2 any ECG abnormality"]
collapsed.columns = ["no disease", "disease"]
chi2_c, p_c, _, exp_c = stats.chi2_contingency(collapsed)
print(collapsed)
print(f"smallest expected count now: {exp_c.min():.1f}")
print(f"chi-square p = {p_c:.4f}   <- trustworthy\n")

# Fix 2: an exact test, which needs no large-count approximation.
p_exact = stats.fisher_exact(restecg_table.to_numpy()[[0, 2]])[1]
print(f"Fisher's exact on the two well-populated rows: p = {p_exact:.4f}")

# Fix 3: don't test it. 4 patients cannot support a conclusion either way.
print(f"\npatients in restecg category 1: {(df['restecg'] == 1).sum()}")

**Observe:** collapsing categories 1 and 2 into "any ECG abnormality" lifts the smallest
expected count well clear of 5 and yields a valid p-value; the exact test on the two
well-populated rows agrees closely.
**Infer:** collapsing is the right first move *when the merged categories are
clinically coherent* — ST-T abnormality and LV hypertrophy are both abnormal ECG
findings, so merging them answers a real question ("does an abnormal ECG associate with
disease?") rather than manufacturing a category to rescue a test. Merging unrelated
categories to make the arithmetic work is data dredging with extra steps. Fix 3 deserves
equal weight: with four patients, every option is an estimate built on four patients,
and Session 6's power analysis says nothing here could detect anything but an enormous
effect. Reporting "insufficient data for this category" is a legitimate finding, and a
more honest one than a rescued p-value.

## Step 6 — Effect size: Cramér's V

Like chi-square itself, the statistic grows with sample size. **Cramér's V** normalises
it to [0, 1], making associations comparable across tables of different sizes — the
categorical counterpart to Session 7's Cohen's d.

In [ ]:
import numpy as np

def cramers_v(chi2_stat, table):
    n = table.to_numpy().sum()
    r, k = table.shape
    return np.sqrt((chi2_stat / n) / min(r - 1, k - 1))

print(f"{'input':10} {'chi2':>8} {'Cramer V':>10} {'strength':>10}")
for name, table in [("cp", contingency), ("exang", exang_table), ("restecg", collapsed)]:
    c2 = stats.chi2_contingency(table)[0]
    v = cramers_v(c2, table)
    strength = "strong" if v >= 0.5 else ("moderate" if v >= 0.3 else ("weak" if v >= 0.1 else "negligible"))
    print(f"{name:10} {c2:8.2f} {v:10.3f} {strength:>10}")

**Observe:** `cp` at `V = 0.510` (strong), `exang` at `0.414` (moderate), and the
collapsed `restecg` far below both.
**Infer:** V reorders nothing here relative to the p-values, but it changes what you can
*say*: "cp has a strong association, restecg a weak one" is a comparison the raw
chi-square statistics cannot support, since `cp`'s `77.28` and `exang`'s `50.94` come
from tables of different shapes. The reordering matters more when sample sizes differ
between tables — a large table can produce a bigger chi-square from a weaker
association, purely on volume. Report V (or the odds ratio for 2×2) alongside every
chi-square, for the same reason Session 6 insisted on effect size alongside every
p-value.

## Step 7 — Residual analysis: which category drives the association?

A significant chi-square says the table as a whole departs from independence. It does
not say where. **Standardised residuals**, $(O - E)/\sqrt{E}$, localise it: beyond
about ±2 marks a cell contributing substantially.

In [ ]:
observed = contingency.to_numpy()
residuals = (observed - expected) / np.sqrt(expected)
residual_table = pd.DataFrame(residuals, index=contingency.index, columns=contingency.columns)
print("standardised residuals (O - E) / sqrt(E):")
print(residual_table.round(2))
print("\ncells beyond +/-2 (substantial contributors):")
print((residual_table.abs() > 2).sum().sum(), "of", residual_table.size)

contribution = (residuals ** 2) / chi2
print("\nshare of the total chi-square from each row:")
print(pd.Series(contribution.sum(axis=1), index=contingency.index).round(3).to_string())

**Observe:** six of eight cells exceed ±2, but the asymptomatic row carries the largest
residuals (`−4.29` and `+4.63`) and `0.516` of the total statistic on its own — more
than the other three rows combined — while typical angina contributes `0.029`.
**Infer:** the association is concentrated, not spread evenly — practically, `cp` is
useful because it identifies asymptomatic patients, and a binary "asymptomatic yes/no"
would capture most of its predictive value with a quarter of the encoding cost. That is
a concrete input to Session 9's design: four dummy variables versus one, on 297
patients, is a meaningful difference in how much data each estimated coefficient gets.
The sign pattern is the readable part — negative in "no disease", positive in
"disease", meaning asymptomatic patients are over-represented among the diseased
relative to independence. Session 5's correlation, by contrast, could only report one
number for the whole variable and could not have located this.

## Step 8 — Screen every categorical input at once

Session 6's multiple-comparisons discipline, applied to the categorical half of the
registry.

In [ ]:
from statsmodels.stats.multitest import multipletests

categorical = ["sex", "cp", "fbs", "restecg", "exang", "slope", "thal", "ca"]

rows = []
for col in categorical:
    table = pd.crosstab(df[col], df["target"])
    c2, p, dof_c, exp_c2 = stats.chi2_contingency(table)
    rows.append({
        "input": col,
        "categories": table.shape[0],
        "chi2": c2,
        "p": p,
        "Cramer V": cramers_v(c2, table),
        "min expected": exp_c2.min(),
        "valid?": exp_c2.min() >= 5,
    })

screen = pd.DataFrame(rows).sort_values("Cramer V", ascending=False)
screen["BH significant"] = multipletests(screen["p"], alpha=ALPHA, method="fdr_bh")[0]
print(screen.round({"chi2": 2, "p": 6, "Cramer V": 3, "min expected": 2}).to_string(index=False))

**Observe:** `thal`, `cp`, `ca`, and `exang` lead on Cramér's V and clear BH correction;
`fbs` is negligible on every column; and the `valid?` flag is `False` for `restecg` —
and for any other input with a sparse category.
**Infer:** sorting by effect size rather than p-value puts the table in the order a
modeller actually needs, and it agrees with Session 5's correlation ranking and Session
7's Cohen's d ranking for the inputs those covered — four methods converging on the same
shortlist. The `valid?` column is the one that earns its place: it is easy to sort a
table like this by p and never notice that one row's p-value is not valid at all, which
is precisely the Step 5 failure at scale. Any row flagged `False` needs collapsing or an
exact test before its number counts.

## What this session hands to the next one

- **A tested set of categorical predictors**, ranked by Cramér's V rather than p-value:
  `thal`, `cp`, `ca`, `exang` strong-to-moderate, `fbs` negligible.
- **An encoding recommendation** from Step 7: `cp`'s signal is concentrated in the
  asymptomatic category, so a binary indicator may serve Session 9 better than four
  dummies.
- **A validity flag** on `restecg`, which must be collapsed before use.
- **Completed feature screening.** Sessions 5-8 together have taken thirteen raw
  columns and produced a ranked, tested, effect-sized shortlist.

Session 9 stops screening and starts building: from associations one input at a time, to
a model that combines them.

## Try it yourself

1. Rebuild Step 2's table with `cp` collapsed to "asymptomatic vs. anything else". How
   much of the original chi-square and Cramér's V survives the simplification?
2. Run Step 7's residual analysis on `thal`. Which category drives its association, and
   does the same "one category does the work" pattern hold?
3. Bin `age` into Session 1's four groups and add it to the Step 8 screen. Does the
   binned version's Cramér's V justify losing the continuous resolution?
4. Take a 60-patient subsample and re-run Step 8. How many inputs now fail the
   `valid?` check, and which conclusions from the full registry survive?